In [0]:
%pip install -U --quiet databricks-sdk "databricks-langchain" databricks-agents mlflow[databricks] databricks-vectorsearch langchain langchain_core bs4 markdownify pydantic mlflow openai PyMuPDF
dbutils.library.restartPython()


In [0]:
from databricks_langchain import ChatDatabricks
 
chat_model = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    temperature=0.1,
    max_tokens=250,
)
chat_model.invoke("What is Databricks Vector Database?")

In [0]:
chat_model.invoke("What is Agentcore Knowledge?")

In [0]:
#Bedrock agentcore PDF Example.
source_pdf = "/Volumes/workspace/default/samplevectorstore/Amazon+Bedrock+AgentCore_v1.1_30112025.pdf"
 
 
 
import fitz  # PyMuPDF
 
# Read file content
with open(source_pdf, "rb") as f:
    pdf_bytes = f.read()
 
# Use PyMuPDF to extract text
doc = fitz.open("pdf", pdf_bytes)
 
text = ""
for page in doc:
    text += page.get_text()
 
print(text)  # Print first 1000 characters
 
 
# CSV Example
# df = spark.read.option("header", "true").csv("/Volumes/<catalog>/<schema>/<volume>/filename.csv")
 
# OR Parquet
# df = spark.read.parquet("/Volumes/<catalog>/<schema>/<volume>/filename.parquet")
 
# OR JSON
# df = spark.read.option("multiline", "true").json("/Volumes/<catalog>/<schema>/<volume>/filename.json")


In [0]:

import fitz  # PyMuPDF
 
# Read file content
with open(source_pdf, "rb") as f:
    pdf_bytes = f.read()
 
# Use PyMuPDF to extract text
doc = fitz.open("pdf", pdf_bytes)
 
text = ""
for page in doc:
    text += page.get_text()
 
print(text)  # Print first 1000 characters
 
# CSV Example
# df = spark.read.option("header", "true").csv("/Volumes/<catalog>/<schema>/<volume>/filename.csv")
 
# OR Parquet
# df = spark.read.parquet("/Volumes/<catalog>/<schema>/<volume>/filename.parquet")
 
# OR JSON
# df = spark.read.option("multiline", "true").json("/Volumes/<catalog>/<schema>/<volume>/filename.json")
 
 
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
 
# Load documents (e.g., text files or PDFs from DBFS or local)
raw_text = text
 # Replace this with your actual file reader logic
 
# Chunk documents for embedding
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
docs = text_splitter.create_documents([raw_text])
 
len(docs)
docs[0]
docs[1]
 
import pandas as pd# Convert docs to a list of dicts for display
pd_docs = pd.DataFrame([doc.dict() for doc in docs])
 
pd_docs.insert(0, "id_pk", range(1, len(pd_docs) + 1))
 
display(pd_docs)

In [0]:
spark_df = spark.createDataFrame(pd_docs[['id_pk','page_content']])
spark_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.my_data_chunks")

In [0]:
from databricks.vector_search.client import VectorSearchClient
 
client = VectorSearchClient()
 
index=client.get_index(index_name="workspace.default.myindex")
 
results_dict=index.similarity_search(
     query_text="What is AgentCore Runtime?",
     columns=["id_pk", "page_content"],
     num_results=2
 )
 
display(results_dict)

In [0]:
from databricks_langchain import ChatDatabricks

chat_model = ChatDatabricks(
    endpoint = "databricks-meta-llama-3-3-70b-instruct",
    temperature=0.1,
    max_tokens=250,
)
 
chat_model.invoke("Give me all the AgentCore Services")

In [0]:
%pip install langchain databricks-vectorsearch pypdf langchain-community

dbutils.library.restartPython()

In [0]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from databricks_langchain import DatabricksEmbeddings
import pandas as pd

loader = PyPDFDirectoryLoader("/Volumes/workspace/default/studebakerrepairs")
docs = loader.load()
print(f"Length of Docs {len(docs)}")
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(docs)

embedding_model = DatabricksEmbeddings(
    endpoint="databricks-bge-large-en"
)

# docs = pd.DataFrame([doc.dict() for doc in docs])
# docs.insert(0, "id_pk", range(1, len(pd_docs) + 1))
 
df = pd.DataFrame({
    "text": [c.page_content for c in chunks]
})
df.insert(0, "id_pk", range(1, len(df)+1))
spark_df = spark.createDataFrame(df)

spark_df.write.format("delta").saveAsTable(
    "workspace.default.studebaker_embeddings"
)

In [0]:
import fitz
import os
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
from databricks.vector_search.client import VectorSearchClient

volume_path = "/Volumes/workspace/default/studebakerrepairs"

# Walk the volume and OCR every PDF page
all_texts = []
for root, dirs, files in os.walk(volume_path):
    for fname in sorted(files):
        if not fname.lower().endswith(".pdf"):
            continue
        fpath = os.path.join(root, fname)
        short = os.path.relpath(fpath, volume_path)
        doc = fitz.open(fpath)
        page_count = len(doc)
        print(f"Processing {short} ({page_count} pages)...")
        for i, page in enumerate(doc):
            tp = page.get_textpage_ocr(language="eng", dpi=300, full=True)
            text = page.get_text(textpage=tp).strip()
            if text:
                all_texts.append({"source": short, "page": i, "text": text})
        doc.close()
        print(f"  -> extracted text from {sum(1 for t in all_texts if t['source'] == short)} pages")

print(f"\nTotal pages with text: {len(all_texts)}")

# Chunk the extracted text
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunked = []
for entry in all_texts:
    chunks = splitter.split_text(entry["text"])
    for chunk in chunks:
        chunked.append({"source": entry["source"], "page": entry["page"], "text": chunk})

print(f"Total chunks: {len(chunked)}")

# Save to Delta table (overwrite)
df = pd.DataFrame(chunked)
df.insert(0, "id_pk", range(1, len(df) + 1))
spark_df = spark.createDataFrame(df)
spark_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.default.studebaker_embeddings"
)
print(f"Wrote {len(df)} rows to workspace.default.studebaker_embeddings")

# Trigger sync on the existing vector search index
client = VectorSearchClient(disable_notice=True)
index = client.get_index(index_name="workspace.default.studebaker_vector_search")
index.sync()
print("Triggered index sync on workspace.default.studebaker_vector_search")

In [0]:
display(spark_df)

In [0]:
from databricks.vector_search.client import VectorSearchClient
import time

client = VectorSearchClient(disable_notice=True)

# Create a Delta Sync Index on the studebaker_embeddings source table.
# Databricks computes embeddings automatically using the specified model.
index = client.create_delta_sync_index(
    endpoint_name="vectorindex",
    source_table_name="workspace.default.studebaker_embeddings",
    index_name="workspace.default.studebaker_index",
    pipeline_type="TRIGGERED",
    primary_key="id_pk",
    embedding_source_column="text",
    embedding_model_endpoint_name="databricks-bge-large-en"
)

# Wait for the index to be ready before querying
while not index.describe().get("status", {}).get("ready", False):
    message = index.describe().get("status", {}).get("message", "")
    print(f"Index not ready: {message}. Waiting 30s...")
    time.sleep(30)

print("Index is ready!")

In [0]:
from databricks.vector_search.client import VectorSearchClient
 
client = VectorSearchClient()

index=client.get_index(index_name="workspace.default.studebaker_vector_search")
 
results_dict=index.similarity_search(
     query_text="How do I repair a 3E11 Studebaker Truck Windshield Wiper?",
     columns=["id_pk", "text"],
     num_results=2
 )
 
display(results_dict)

In [0]:
from databricks_langchain import DatabricksVectorSearch

vector_store = DatabricksVectorSearch(index_name="workspace.default.studebaker_vector_search")

retriever = vector_store.as_retriever(search_kwargs={"k":2})

answer = retriever.invoke("How do I repair the windshield wiper motor on a Studebaker Truck?")
answer